# 04 — Base Detection

## Goal
Identify **base clusters** — consecutive candles where price is consolidating rather than trending.  
A base is the "loading zone" between two legs; it's where institutional orders are placed.

## What Makes a Candle a "Base" Candle?
One condition, already computed in NB02 and stored in the `is_base` column:

$$\text{body\_ratio} = \frac{|\text{close} - \text{open}|}{\text{high} - \text{low}} \leq 0.50$$

A candle qualifies if its body is at most 50% of its total range — meaning it closed near where it opened (no strong directional push). NB04 does **not** re-filter `is_base`; it only clusters the candles that already carry `is_base = True`.

## Cluster Filters
Once a contiguous run of base candles is found, it must pass **tightness gates** at the cluster level:

| Filter | Equation | Threshold |
|--------|----------|-----------|
| Min length | $\text{count} \geq 1$ | At least 1 candle |
| Max length | $\text{count} \leq 5$ | No more than 5 candles |
| Width / ATR | $\dfrac{\text{base\_high} - \text{base\_low}}{\text{avg\_ATR}} \leq 2.5$ | Cluster height is tight |
| Compactness | $\dfrac{\text{width\_atr\_ratio}}{\text{count}} \leq 0.80$ | Tight relative to its length |

## Known Limitation — Scenario D
Scenario D contains a wide-range candle with a small body (`body_ratio ≈ 0.145`). It passes the `is_base` check and forms a cluster that also passes the tightness gates, so the algorithm **incorrectly detects D as a zone** (false positive).

The body-ratio gate alone cannot distinguish a calm narrow candle from a volatile doji on a 5-point swing. A future fix would add a per-candle `range / ATR ≤ 1.5` pre-filter to `is_base`. For now, Scenario D is a known false positive visible in NB07.

## Visualization
The chart highlights all detected base clusters as shaded boxes overlaid on the candlestick chart. Green = passed all gates; red dotted = rejected.


## 1. Set up thresholds and load the enriched dataset
Each threshold below maps directly to one of the gates from the intro. Keeping them as named constants (not magic numbers buried in code) means every decision the detector makes is traceable to one line you can tune.

| Constant | Meaning |
|----------|---------|
| `BASE_BODY_RATIO_MAX` | A candle qualifies as "base" if its body covers at most 50% of its range |
| `BASE_MIN_CANDLES` | A cluster needs at least this many candles |
| `BASE_MAX_CANDLES` | A cluster stops growing past this length |
| `BASE_MAX_ATR_WIDTH` | Cluster height (high − low) divided by ATR must stay below this |
| `BASE_COMPACTNESS_MAX` | Width-per-candle ratio — penalizes wide clusters that span many bars |


In [21]:
import pandas as pd

BASE_BODY_RATIO_MAX: float = 0.50    # body / range  (set in NB02)
BASE_MIN_CANDLES: int     = 1        # minimum cluster length
BASE_MAX_CANDLES: int     = 5        # maximum cluster length
BASE_MAX_ATR_WIDTH: float = 2.5      # (base_high − base_low) / ATR  ≤ this
BASE_COMPACTNESS_MAX: float = 0.80   # (close_max − close_min) / base_width  ≤ this

df = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
print(f"Loaded {len(df)} candles  |  is_base = {df['is_base'].sum()}/{len(df)}")
df.head()


Loaded 98 candles  |  is_base = 64/98


,open,high,low,close,volume,range,body,body-ratio,is_base,is_doji,is_bullish,atr
Date,,,,,,,,,,,,
2024-01-01 00:00:00+00:00,100.0,101.1,99.5,100.6,1000000,1.6,0.6,0.375000,True,False,True,1.600000
2024-01-08 00:00:00+00:00,100.6,101.1,99.6,100.1,1000000,1.5,0.5,0.333333,True,False,False,1.592857
2024-01-15 00:00:00+00:00,100.1,101.2,99.6,100.7,1000000,1.6,0.6,0.375000,True,False,True,1.593367
2024-01-22 00:00:00+00:00,100.7,101.2,99.7,100.2,1000000,1.5,0.5,0.333333,True,False,False,1.586698
2024-01-29 00:00:00+00:00,100.2,101.3,99.7,100.8,1000000,1.6,0.6,0.375000,True,False,True,1.587648


## 2. Find base clusters
A base **cluster** is a contiguous run of candles where each one already carries `is_base = True` (computed in NB02).

The walking rule is simple:
1. Scan candles left-to-right.
2. When you land on a base candle, **start a cluster**.
3. **Extend** the cluster as long as the next candle is also base *and* you haven't exceeded `BASE_MAX_CANDLES`.
4. When extension stops, save the cluster and resume scanning from the next index.

This is exactly the first half of the production `detect_zones()` function in `zone_detector.py` — same logic, just lifted out so we can inspect it in isolation.


In [22]:
def find_base_clusters(df: pd.DataFrame) -> list[dict]:
    if "is_base" not in df.columns:
        raise ValueError("DataFrame missing 'is_base' column — run NB02 first.")

    clusters = []
    is_base = df["is_base"].to_numpy()
    n = len(df)
    i = 0
    while i < n:
        if not is_base[i]:
            i += 1
            continue

        start = end = i
        while (
            end + 1 < n
            and is_base[end + 1]
            and (end - start + 1) < BASE_MAX_CANDLES
        ):
            end += 1

        clusters.append({"start": start, "end": end, "count": end - start + 1})
        i = end + 1
    return clusters


clusters = find_base_clusters(df)
print(f"Found {len(clusters)} base cluster(s)")
for c in clusters[:15]:
    print(c)


Found 27 base cluster(s)
{'start': 0, 'end': 4, 'count': 5}
{'start': 5, 'end': 9, 'count': 5}
{'start': 10, 'end': 14, 'count': 5}
{'start': 15, 'end': 16, 'count': 2}
{'start': 19, 'end': 21, 'count': 3}
{'start': 23, 'end': 23, 'count': 1}
{'start': 27, 'end': 29, 'count': 3}
{'start': 31, 'end': 33, 'count': 3}
{'start': 35, 'end': 37, 'count': 3}
{'start': 39, 'end': 41, 'count': 3}
{'start': 43, 'end': 43, 'count': 1}
{'start': 46, 'end': 47, 'count': 2}
{'start': 50, 'end': 51, 'count': 2}
{'start': 53, 'end': 54, 'count': 2}
{'start': 57, 'end': 58, 'count': 2}


## 3. Evaluate each cluster — tightness gates
Finding contiguous base candles is only the first step. The cluster as a whole must be **geometrically tight** to qualify as a real consolidation zone. Two measurements decide that:

---

**Gate 1 — Width / ATR: is the zone narrow relative to volatility?**

$$\text{width\_atr\_ratio} = \frac{\text{base\_high} - \text{base\_low}}{\overline{\text{ATR}}} \leq 2.5$$

A cluster whose high-to-low span covers more than 2.5× the local ATR is too wide — price was swinging, not consolidating. This is the primary tightness gate.

---

**Gate 2 — Compactness: are closing prices clustered together?**

$$\text{compactness} = \frac{\text{close\_max} - \text{close\_min}}{\text{base\_width}} \leq 0.80$$

Even if the full range is acceptable, we want the *closing prices* (where candles actually settle) to be tightly grouped. If closes scatter from the very bottom to the very top of the base, the candles aren't in genuine equilibrium — price was just oscillating.

**Intuition:** `compactness = 0` means all candles closed at exactly the same price (perfectly calm). `compactness = 1.0` means closes covered the full width of the zone (not tight at all). We allow up to **0.80**.

---

A cluster passes when all three conditions are met:
- `width_atr_ratio ≤ 2.5`
- `compactness ≤ 0.80`
- `count ≥ BASE_MIN_CANDLES`


In [23]:
def evaluate_cluster(df: pd.DataFrame, cluster: dict) -> dict:
    s, e = cluster["start"], cluster["end"]
    sub = df.iloc[s : e + 1]

    base_high  = sub["high"].max()
    base_low   = sub["low"].min()
    base_width = base_high - base_low
    avg_atr    = sub["atr"].mean()

    # Gate 1: how wide is the zone relative to local volatility?
    width_atr_ratio = base_width / avg_atr

    # Gate 2: how spread are the closing prices within the zone?
    # close_range / base_width  →  0 = all closes identical, 1 = closes cover the full range
    close_range       = sub["close"].max() - sub["close"].min()
    compactness_ratio = close_range / base_width if base_width > 0 else 1.0

    return {
        **cluster,
        "base_high":          round(base_high, 3),
        "base_low":           round(base_low, 3),
        "base_width":         round(base_width, 3),
        "avg_atr":            round(avg_atr, 3),
        "width_atr_ratio":    round(width_atr_ratio, 3),
        "compactness_ratio":  round(compactness_ratio, 3),
        "width_passed":       width_atr_ratio   <= BASE_MAX_ATR_WIDTH,
        "compactness_passed": compactness_ratio <= BASE_COMPACTNESS_MAX,
        "min_count_passed":   cluster["count"]  >= BASE_MIN_CANDLES,
    }


evaluated = [evaluate_cluster(df, c) for c in clusters]
pd.DataFrame(evaluated)


,start,end,count,base_high,base_low,base_width,avg_atr,width_atr_ratio,compactness_ratio,width_passed,compactness_passed,min_count_passed
0,0,4,5,101.3,99.5,1.8,1.592,1.131,0.389,True,True,True
1,5,9,5,101.5,99.8,1.7,1.578,1.077,0.412,True,True,True
2,10,14,5,101.8,100.0,1.8,1.570,1.146,0.389,True,True,True
3,15,16,2,101.8,99.6,2.2,1.551,1.418,0.182,True,True,True
4,19,21,3,105.9,104.9,1.0,1.587,0.630,0.300,True,True,True
5,23,23,1,110.4,109.0,1.4,1.744,0.803,0.000,True,True,True
6,27,29,3,106.7,105.8,0.9,1.756,0.513,0.222,True,True,True
7,31,33,3,102.3,100.2,2.1,1.801,1.166,0.238,True,True,True
8,35,37,3,104.0,103.0,1.0,1.677,0.596,0.200,True,True,True
9,39,41,3,104.4,103.2,1.2,1.440,0.834,0.083,True,True,True


## 4. Walkthrough — Scenario A
Scenario A is the **textbook Rally-Base-Rally** (RBR): an up-leg, then 2–3 calm base candles around the 105 level, then another up-leg.

We locate those candles, find which cluster the detector built around them, and print the full evaluation. Every number you see is what the production code would see at this exact point.


In [24]:
scenario_a_mask = df["close"].between(105.0, 106.0) & df["is_base"]
scenario_a_positions = [df.index.get_loc(idx) for idx in df[scenario_a_mask].index]
print("Candidate base candle positions for Scenario A:", scenario_a_positions)

cluster_a = next(c for c in clusters if c["start"] in scenario_a_positions)
print("\nDetected cluster A:", cluster_a)

result_a = evaluate_cluster(df, cluster_a)
print("\nEvaluation:")
for k, v in result_a.items():
    print(f"  {k}: {v}")

print("\nRaw base candles:")
df.iloc[cluster_a["start"]:cluster_a["end"] + 1][["open", "high", "low", "close", "body-ratio", "is_base", "atr"]]


Candidate base candle positions for Scenario A: [19, 20, 21, 29, 43]

Detected cluster A: {'start': 19, 'end': 21, 'count': 3}

Evaluation:
  start: 19
  end: 21
  count: 3
  base_high: 105.9
  base_low: 104.9
  base_width: 1.0
  avg_atr: 1.587
  width_atr_ratio: 0.63
  compactness_ratio: 0.3
  width_passed: True
  compactness_passed: True
  min_count_passed: True

Raw base candles:


,open,high,low,close,body-ratio,is_base,atr
Date,,,,,,,
2024-05-13 00:00:00+00:00,105.4,105.8,105.0,105.3,0.125,True,1.646234
2024-05-20 00:00:00+00:00,105.3,105.7,104.9,105.5,0.250,True,1.585789
2024-05-27 00:00:00+00:00,105.5,105.9,105.1,105.2,0.375,True,1.529661


## 5. Walkthrough — Scenario D (current false positive)
Scenario D contains two **wide-range candles**: small bodies (`body_ratio ≈ 0.145`) sitting on a 5+ point swing. They pass the `is_base` body-ratio check and can appear in a cluster that also passes the tightness gates — making D a **false positive** in the current pipeline.

Here we:
1. Locate those wide-range candles by their open/close signature.
2. Check whether the detector found any cluster overlapping them.
3. Force-evaluate a cluster built *only* from those two candles to see what `width_atr_ratio` looks like in isolation.

The forced evaluation shows a wide ratio — confirming that **if** the cluster were built from just these two candles it would fail. Whether the real detected cluster for D passes or fails depends on which candles the sliding-window actually groups together.


In [25]:
mask_d = (
    ((df["open"].round(2) == 106.80) & (df["close"].round(2) == 106.00)) |
    ((df["open"].round(2) == 106.00) & (df["close"].round(2) == 110.50))
)
wide_rows = df[mask_d][["open", "high", "low", "close", "range", "body", "body-ratio", "is_base", "atr"]]
print("Scenario D 'wide' candles:")
print(wide_rows)

wide_positions = [df.index.get_loc(idx) for idx in wide_rows.index]
print("\nPositions in df:", wide_positions)

matching = [c for c in clusters if c["start"] in wide_positions or c["end"] in wide_positions]
print("\nClusters overlapping these positions:", matching)


Scenario D 'wide' candles:
                            open   high    low  close  range  body  \
Date                                                                 
2024-10-28 00:00:00+00:00  106.8  110.5  105.0  106.0    5.5   0.8   
2024-11-04 00:00:00+00:00  106.0  111.0  104.5  110.5    6.5   4.5   

                           body-ratio  is_base       atr  
Date                                                      
2024-10-28 00:00:00+00:00    0.145455     True  1.805965  
2024-11-04 00:00:00+00:00    0.692308    False  2.141253  

Positions in df: [43, 44]

Clusters overlapping these positions: [{'start': 43, 'end': 43, 'count': 1}]


In [26]:
forced_cluster = {
    "start": wide_positions[0],
    "end":   wide_positions[-1],
    "count": wide_positions[-1] - wide_positions[0] + 1,
}
print("Forced evaluation (bypassing body-ratio gate):")
forced = evaluate_cluster(df, forced_cluster)
for k, v in forced.items():
    print(f"  {k}: {v}")


Forced evaluation (bypassing body-ratio gate):
  start: 43
  end: 44
  count: 2
  base_high: 111.0
  base_low: 104.5
  base_width: 6.5
  avg_atr: 1.974
  width_atr_ratio: 3.293
  compactness_ratio: 0.692
  width_passed: False
  compactness_passed: True
  min_count_passed: True


## 6. Visualization
Two-panel TradingView-style chart:
- **Top** — candles with each cluster overlaid as a shaded rectangle. Green ✓ = passed all gates, red dotted = failed.
- **Bottom** — ATR line, so you can see *why* a given cluster passed or failed by eyeballing the local volatility.


In [27]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
COLOR_BASE = "#b0bec5"
BG         = "#131722"
GRID       = "#1e222d"

# split into passed / failed once, reuse below
passed = [e for e in evaluated if e["width_passed"] and e["compactness_passed"] and e["min_count_passed"]]
failed = [e for e in evaluated if not (e["width_passed"] and e["compactness_passed"] and e["min_count_passed"])]

print(f"Passed: {len(passed)}   Failed: {len(failed)}")


Passed: 26   Failed: 1


In [28]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],
    vertical_spacing=0.03,
)

fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"],
    low=df["low"],   close=df["close"],
    increasing_line_color=COLOR_BULL,
    decreasing_line_color=COLOR_BEAR,
    name="Price",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df["atr"],
    mode="lines",
    line=dict(color="#7c83fd", width=1.5),
    fill="tozeroy",
    fillcolor="rgba(124,131,253,0.10)",
    name="ATR(14)",
    hovertemplate="%{x}<br>ATR: %{y:.3f}<extra></extra>",
), row=2, col=1)


In [29]:
# overlay passed clusters in green with a ✓ annotation
for cluster in passed:
    x0, x1 = df.index[cluster["start"]], df.index[cluster["end"]]
    fig.add_vrect(
        x0=x0, x1=x1,
        fillcolor="rgba(38,166,154,0.15)",
        line=dict(color="#26a69a", width=1, dash="dot"),
        row=1, col=1,
        annotation_text=f"✓ {cluster['count']}c",
        annotation_font=dict(size=9, color="#26a69a"),
        annotation_position="top left",
    )

# overlay failed clusters in faint red — no annotation, just visible
for cluster in failed:
    x0, x1 = df.index[cluster["start"]], df.index[cluster["end"]]
    fig.add_vrect(
        x0=x0, x1=x1,
        fillcolor="rgba(239,83,80,0.08)",
        line=dict(color="#ef5350", width=1, dash="dot"),
        row=1, col=1,
    )


In [30]:
fig.update_layout(
    title=f"Base Detection — {len(passed)} valid cluster(s)  |  {len(failed)} rejected",
    height=620,
    plot_bgcolor=BG, paper_bgcolor=BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.04),
)
shared_axis = dict(gridcolor=GRID, showgrid=True, zeroline=False, linecolor=GRID)
fig.update_xaxes(**shared_axis)
fig.update_yaxes(**shared_axis)
fig.update_yaxes(title_text="Price",   row=1, col=1)
fig.update_yaxes(title_text="ATR(14)", row=2, col=1)
fig.update_xaxes(showticklabels=True,  row=2, col=1)

fig.show()
